# SlotSaver — Level 3: Fairness audit & drift monitoring

Two questions a responsible deployment must answer continuously:

1. **Fairness** — are the probabilities equally honest for every group, and
   does every group's no-shows get a fair share of the help?
2. **Drift** — is the world still the one the model was trained on?

(The serving side of Level 3 lives in `api.py` + `Dockerfile` +
`nightly_job.py`; this notebook is the analysis side.)

In [1]:
import sys
if "google.colab" in sys.modules:          # this cell does nothing on your PC
    !unzip -q -o slotsaver.zip
    %cd slotsaver
    !mkdir -p data
    !cp /content/noshowappointments*.csv data/KaggleV2-May-2016.csv
    %pip install -q lightgbm

/content/slotsaver


In [2]:
import pandas as pd

from src.calibrate import three_way_temporal_split
from src.data import load_or_synthesize
from src.fairness import add_age_band, audit, flag_concerns
from src.features import add_history_features
from src.monitor import any_alert, drift_report, psi
from src.pipeline import build_scoring_artifacts

pd.set_option("display.max_columns", 20)

In [3]:
art = build_scoring_artifacts("data/KaggleV2-May-2016.csv")
scored = art["scored_test"]

clean(): dropped 6 bad-age rows, 5 negative-lead-time rows (110527 -> 110516)
three_way_temporal_split(): train=67357 (to 2016-05-20)  cal=12431 (2016-05-24..2016-05-30)  test=30728 (from 2016-05-31)


## 1. Fairness audit

Reading guide for each table:

- `calibration_gap` = mean predicted − actual. Near 0 is honest. A group at
  +0.05 is being systematically over-flagged; at −0.05, under-served.
- `benefit_rate` = of this group's *actual* no-shows, the share placed in the
  staff_call tier (strongest help). Large gaps between groups = unequal help.
- `call_tier_share` vs `population_share` = over/under-representation on the
  call list. Differing base rates can justify differences — but you must be
  able to say *why*, out loud, before shipping.

In [4]:
tables = audit(scored)
tables["gender"]

,n,population_share,actual_noshow_rate,mean_predicted_p,calibration_gap,auc_within_group,call_tier_share,benefit_rate
gender,,,,,,,,
F,20112,0.655,0.183,0.199,0.016,0.722,0.644,0.218
M,10616,0.345,0.186,0.197,0.011,0.734,0.356,0.225


In [5]:
tables["scholarship"]

,n,population_share,actual_noshow_rate,mean_predicted_p,calibration_gap,auc_within_group,call_tier_share,benefit_rate
scholarship,,,,,,,,
0,27733,0.903,0.181,0.196,0.015,0.720,0.842,0.200
1,2995,0.097,0.211,0.216,0.005,0.768,0.158,0.383


In [6]:
tables["age_band"]

,n,population_share,actual_noshow_rate,mean_predicted_p,calibration_gap,auc_within_group,call_tier_share,benefit_rate
age_band,,,,,,,,
0-12,5781,0.188,0.182,0.205,0.023,0.724,0.220,0.217
13-25,4720,0.154,0.247,0.237,-0.010,0.731,0.279,0.358
26-45,7934,0.258,0.199,0.205,0.006,0.728,0.298,0.234
46-65,8428,0.274,0.154,0.179,0.026,0.691,0.150,0.137
65+,3865,0.126,0.144,0.166,0.022,0.670,0.053,0.097


### Automatic flags

Thresholds (|calibration_gap| > 0.05, min/max benefit ratio < 0.5) are review
triggers, not verdicts — every flag demands a human explanation.

In [7]:
concerns = flag_concerns(tables)
if concerns:
    for c in concerns:
        print("⚠", c)
else:
    print("No automatic flags at current thresholds — still read the tables; "
          "thresholds are conventions, not guarantees.")

⚠ age_band: benefit_rate ranges 0.10–0.36 — some groups' no-shows get far less help than others


## 2. Drift monitoring

PSI compares distributions against the training reference. Thresholds:
< 0.10 stable · 0.10–0.25 watch · > 0.25 alert. First: the honest check of
our own test window against training — some drift is already visible in any
real system.

In [8]:
df, _ = load_or_synthesize("data/KaggleV2-May-2016.csv")
df = add_history_features(df)
train, cal, test = three_way_temporal_split(df)

report = drift_report(train, test)
print(report.to_string(index=False))
print("\nRetrain trigger fires:", any_alert(report))

clean(): dropped 6 bad-age rows, 5 negative-lead-time rows (110527 -> 110516)
three_way_temporal_split(): train=67357 (to 2016-05-20)  cal=12431 (2016-05-24..2016-05-30)  test=30728 (from 2016-05-31)
             check   value status
PSI lead_time_days  0.0148     ok
           PSI age  0.0022     ok
   base-rate delta -0.0270  watch

Retrain trigger fires: False


### Simulated future drift — what an alert looks like

Suppose the clinic changes its booking policy and lead times stretch by two
weeks. Watch PSI catch it:

In [9]:
shifted = test.copy()
shifted["lead_time_days"] = shifted["lead_time_days"] + 14

print(f"PSI lead_time (normal window) : {psi(train['lead_time_days'], test['lead_time_days']):.4f}")
print(f"PSI lead_time (+14d shift)    : {psi(train['lead_time_days'], shifted['lead_time_days']):.4f}  <- ALERT")

PSI lead_time (normal window) : 0.0148
PSI lead_time (+14d shift)    : 9.4115  <- ALERT


When this alert fires in `nightly_job.py`, the playbook is: retrain the
model, refit the calibrator (calibration breaks FIRST under drift), and
re-check which constraint — capacity or threshold — now binds.


## 3. Conclusions

Based on the analysis performed in this notebook, here are the key takeaways for the responsible deployment of SlotSaver:

### Fairness Audit Findings
- **Calibration**: The `46-65` and `0-12` age bands show the largest positive calibration gaps (~0.023–0.026), meaning the model slightly over-predicts their no-show probability. While below the 0.05 alert threshold, these groups are being 'over-flagged' for intervention relative to their actual behavior.
- **Benefit Distribution**: There is a significant disparity in `benefit_rate` across age bands. The `13-25` group has a benefit rate of 0.36, while the `65+` group is at only 0.10. This indicates that younger no-shows are much more likely to be placed in the high-touch 'staff_call' tier than elderly no-shows. This warrants a review of whether the model features (like 'lead time') are unfairly penalizing specific demographics.

### Drift & Reliability
- **Baseline Stability**: The current test window shows stable PSI for features like `age` and `lead_time_days`, though the `base-rate delta` sits at 'watch' status, suggesting a slight shift in the overall no-show frequency.
- **Drift Sensitivity**: The simulation demonstrates that a 14-day shift in booking lead times causes the PSI to spike to >9.0, triggering an immediate alert.
- **Operational Strategy**: Calibration is the most fragile part of the pipeline under drift. When drift alerts fire, the immediate playbook must include refitting the Isotonic Regressor to ensure the probabilities shown to clinic staff remain 'honest' even if the underlying feature distributions have changed.